In [ ]:
from pathlib import Path
from IPython.display import display, Markdown

from resources.imports import *
import torch

from resources.MLmetrics import (
    postprocess_list_runs,
    postprocess_artifact_table,
    postprocess_load_field_run,
    postprocess_field_run_overview,
    display_field_run_overview,
    postprocess_build_active_field_diagnostics,
    field_summary_table,
    print_field_diagnostics,
    plot_field_frame_component_trends,
    plot_field_frame_component_heatmaps,
    plot_field_diversity,
    display_field_sample_error_summary,
    display_field_node_error_summary,
    field_sample_viewer,
    plot_field_sample_frame_evolution,
    plot_field_frame_component_collapse,
    plot_field_component_parity,
    plot_field_keyframe_strip,
    plot_loss_history,
)

plt.rcParams.update({
    "figure.figsize": (10, 5),
    "axes.grid": False,
    "axes.spines.top": False,
    "axes.spines.right": False,
})


In [ ]:
%load_ext autoreload
%autoreload 2

# ML Field Post-Processing

## 1. Find Recent Runs

In [ ]:
FIND_RUN = False
RUN_ROOT = Path(r"Z:/p2")

if FIND_RUN:
    recent_runs = postprocess_list_runs(RUN_ROOT, max_runs=25, include_hpo=True)
    display(recent_runs)
    if recent_runs.empty:
        raise ValueError("No saved runs were found. Set RUN_PATH manually.")

## 2. User Configuration

Set the run identity here. `run_type` controls whether the path points to a standard run, a model-specific HPO run, or one model inside a cross-model HPO run.

In [ ]:
mechMode = "UT"
model = "Transformer"
run_name = "TR-Field-UT-1"
VIEW_MODE = None
run_type = "standard"  # "standard", "model_hpo", or "cross_model_hpo"
RUN_PATH_OVERRIDE = None

if RUN_PATH_OVERRIDE is not None:
    RUN_PATH = Path(RUN_PATH_OVERRIDE)
elif run_type == "standard":
    RUN_PATH = RUN_ROOT / mechMode / model / run_name
elif run_type == "model_hpo":
    RUN_PATH = RUN_ROOT / mechMode / model / "HPO" / run_name
elif run_type == "cross_model_hpo":
    RUN_PATH = RUN_ROOT / mechMode / "HPO" / run_name / model
else:
    raise ValueError("run_type must be 'standard', 'model_hpo', or 'cross_model_hpo'.")

DATA_PATH_OVERRIDE = None

if VIEW_MODE is None:
    VIEW_MODE = mechMode if str(mechMode).upper() in ["UT", "FT"] else "UT"
VIEW_MODE = str(VIEW_MODE).upper()
if VIEW_MODE not in ["UT", "FT"]:
    raise ValueError("VIEW_MODE must be UT or FT. For MULTI runs, choose which output branch to view.")

LOAD_DATA = True
LOAD_MODEL = True
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

ACTIVE_SPLIT = None


## 3. Load Run Artifacts

In [ ]:
artifacts, loaded, DAT, MOD = postprocess_load_field_run(
    RUN_PATH,
    run_root=RUN_ROOT,
    load_data=LOAD_DATA,
    load_model=LOAD_MODEL,
    data_path_override=DATA_PATH_OVERRIDE,
    device=DEVICE,
)


display(postprocess_artifact_table(artifacts))
for warning in artifacts.get("warnings", []):
    print("WARNING:", warning)


## 4. Run Setup And Available Outputs

In [ ]:
overview = postprocess_field_run_overview(
    artifacts,
    loaded,
    data=DAT,
    run_name=run_name,
    run_type=run_type,
    mech_mode=mechMode,
    view_mode=VIEW_MODE,
    model_name=model,
    device=DEVICE,
    active_split=ACTIVE_SPLIT,
)


available_evals = overview["available_evals"]
available_field_evals = overview["available_field_evals"]
ACTIVE_SPLIT = overview["active_split"]

display_field_run_overview(overview)
print("mechMode:", mechMode)
print("VIEW_MODE:", VIEW_MODE)
print("ACTIVE_SPLIT:", ACTIVE_SPLIT)


## 5. Build Active Field Diagnostics

In [ ]:
diagnostics, active_key, active_diag = postprocess_build_active_field_diagnostics(
    DAT,
    loaded,
    available_evals,
    view_mode=VIEW_MODE,
    active_split=ACTIVE_SPLIT,
    model=MOD,
)


if active_diag is None:
    print("No active diagnostics are available. Check predictions.npz, diagnostic CSVs, VIEW_MODE, or ACTIVE_SPLIT.")
else:
    print_field_diagnostics(active_diag, label=f"{active_key[0]} {active_key[1]}")


## 6. Active Diagnostic Summary

In [ ]:
SUMMARY_METRICS = [
    "rmse",
    "mae",
    "mse",
    "bias",
    "collapse_ratio",
    "mean_field_baseline_rmse",
    "skill_vs_mean_field_rmse",
    "valid_fraction",
    "n_samples",
    "n_frames",
    "n_nodes",
    "n_components",
]


if active_diag is None:
    print("No active diagnostics are available.")
else:
    display(field_summary_table(active_diag, metrics=SUMMARY_METRICS))


## 7. Frame And Component Performance

In [ ]:
FRAME_TRENDS_FIGSIZE = (14, 8)
HEATMAP_FIGSIZE = (17, 5)
HEATMAP_CMAPS = {"RMSE": "viridis", "Bias": "coolwarm", "Valid Fraction": "magma"}
SHOW_COMPONENT_TABLE = True


if active_diag is None:
    print("No active diagnostics are available.")
else:
    plot_field_frame_component_trends(active_diag, figsize=FRAME_TRENDS_FIGSIZE)
    plot_field_frame_component_heatmaps(active_diag, figsize=HEATMAP_FIGSIZE, cmaps=HEATMAP_CMAPS)

    component_metrics = active_diag.get("component_metrics")
    if SHOW_COMPONENT_TABLE and hasattr(component_metrics, "copy"):
        display(Markdown("### Component Metrics"))
        display(component_metrics)


## 8. Prediction Collapse And Field Diversity

In [ ]:
DIVERSITY_FIGSIZE = (16, 4)
DIVERSITY_RATIO_REFERENCE = 1.0
DIVERSITY_BINS = 40
COLLAPSE_HEATMAP_FIGSIZE = (8, 5)
FIELD_PARITY_COMPONENTS = None
FIELD_PARITY_MAX_POINTS_PER_COMPONENT = 300000
FIELD_PARITY_GRIDSIZE = 70
FIELD_PARITY_FIGSIZE = None


if active_diag is None:
    print("No active diagnostics are available.")
else:
    try:
        plot_field_diversity(
            active_diag,
            figsize=DIVERSITY_FIGSIZE,
            ratio_reference=DIVERSITY_RATIO_REFERENCE,
            bins=DIVERSITY_BINS,
        )
        plot_field_frame_component_collapse(
            active_diag,
            figsize=COLLAPSE_HEATMAP_FIGSIZE,
        )
        plot_field_component_parity(
            active_diag,
            components=FIELD_PARITY_COMPONENTS,
            max_points_per_component=FIELD_PARITY_MAX_POINTS_PER_COMPONENT,
            gridsize=FIELD_PARITY_GRIDSIZE,
            figsize=FIELD_PARITY_FIGSIZE,
        )
    except ValueError as exc:
        print(exc)


## 9. Sample-Level Error

In [ ]:
SAMPLE_DISTRIBUTION_BINS = 40
SAMPLE_DISTRIBUTION_NCOLS = 3
SAMPLE_TABLE_TOP_N = 5
SAMPLE_DESCRIBE_COLUMNS = ["sample_mae", "sample_mse", "sample_rmse", "sample_bias", "valid_fraction"]


display_field_sample_error_summary(
    active_diag,
    bins=SAMPLE_DISTRIBUTION_BINS,
    ncols=SAMPLE_DISTRIBUTION_NCOLS,
    top_n=SAMPLE_TABLE_TOP_N,
    columns=SAMPLE_DESCRIBE_COLUMNS,
)


## 10. Node-Level Spatial Error

In [ ]:
NODE_METRIC = "rmse"
NODE_TABLE_TOP_N = 20
NODE_PLOT_COLUMNS = ["rmse", "mae", "bias", "mean_abs_percent_error"]
NODE_POINT_SIZE = 22


display_field_node_error_summary(
    active_diag,
    metric=NODE_METRIC,
    top_n=NODE_TABLE_TOP_N,
    plot_columns=NODE_PLOT_COLUMNS,
    point_size=NODE_POINT_SIZE,
)


## 11. Interactive Field Viewer

In [ ]:
FIELD_SAMPLE_MODE = "selected"
FIELD_SELECTED_SAMPLES = 0
FIELD_VIEW_FRAME = 10
FIELD_VIEW_COMPONENT = "U2"
FIELD_RANKING_METRIC = "rmse"
FIELD_PLOT_STYLE = "continuous"
FIELD_RANDOM_COUNT = 5
FIELD_KEYFRAME_FRAMES = None
FIELD_KEYFRAME_COUNT = 5
FIELD_KEYFRAME_ROWS = ("truth", "prediction", "error")
FIELD_KEYFRAME_PLOT_STYLE = "continuous"


plot_field_keyframe_strip(
    active_diag,
    sample=FIELD_SELECTED_SAMPLES,
    component=FIELD_VIEW_COMPONENT,
    frames=FIELD_KEYFRAME_FRAMES,
    n_keyframes=FIELD_KEYFRAME_COUNT,
    rows=FIELD_KEYFRAME_ROWS,
    plot_style=FIELD_KEYFRAME_PLOT_STYLE,
)

field_sample_viewer(
    active_diag,
    sample_mode=FIELD_SAMPLE_MODE,
    selected_samples=FIELD_SELECTED_SAMPLES,
    frame=FIELD_VIEW_FRAME,
    component=FIELD_VIEW_COMPONENT,
    ranking_metric=FIELD_RANKING_METRIC,
    plot_style=FIELD_PLOT_STYLE,
    random_count=FIELD_RANDOM_COUNT,
)


## 12. Selected Sample Frame Evolution

In [ ]:
FRAME_EVOLUTION_SAMPLE = 100


plot_field_sample_frame_evolution(active_diag, sample=FRAME_EVOLUTION_SAMPLE)


## 13. Training Loss History

In [ ]:
LOSS_HISTORY_METRICS = ["train_loss", "val_loss"]
LOSS_HISTORY_FIGSIZE = (9, 4)


loss_history = loaded.get("loss_history")
if loss_history is not None and hasattr(loss_history, "empty") and not loss_history.empty:
    display(loss_history.head())
plot_loss_history(loss_history, metrics=LOSS_HISTORY_METRICS, figsize=LOSS_HISTORY_FIGSIZE)
